# Fase 2: ETL Completo (Inyección de 11,500 registros)

Este notebook contiene la lógica completa y final que se utilizó para poblar tu base de datos SQL Server. Genera la información cruzada (Clima + Energía + Economía) y la inserta automáticamente en tu esquema estrella respetando las Llaves Foráneas (FK).

In [ ]:
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Configuración de Conexión a SQL Server
SERVER_NAME = 'LUIS'  # El nombre de tu servidor
DATABASE_NAME = 'DB_Analitica_Predictiva'
conn_str = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;"

print(f"Configuración lista para conectar a: {SERVER_NAME}")

### 1. Generación de Datos (Extraídos y Transformados)
Generamos un dataframe maestro con la información consolidada.

In [ ]:
def generar_dataset_maestro(n_registros=11500):
    np.random.seed(42)
    fechas = [datetime(2023, 1, 1) + timedelta(days=x) for x in range(n_registros)]
    provincias = ['Pichincha', 'Guayas', 'Azuay', 'Manabí', 'Tungurahua', 'Loja']
    sectores = ['Industrial', 'Comercial', 'Residencial', 'Manufactura', 'Público']
    empresas = ['EEQ', 'CNEL Guayas', 'Centrosur', 'CNEL Manabí', 'EEASA', 'EERSSA']
    
    datos = []
    for i in range(n_registros):
        fecha = fechas[i]
        es_estiaje = 1 if fecha.month in [10, 11, 12] else 0
        precipitacion = max(0, np.random.normal(loc=15 - (es_estiaje*10), scale=5))
        temperatura = np.random.normal(loc=22, scale=4)
        
        provincia_idx = random.randint(0, len(provincias)-1)
        sector = random.choice(sectores)
        
        energia_facturada = max(100, np.random.normal(loc=500, scale=100))
        demanda_maxima = energia_facturada * (1.2 + (es_estiaje * 0.15))
        porcentaje_perdidas = np.random.normal(loc=8, scale=2) + (es_estiaje * 1.5)
        
        vab_base = 50000 if sector == 'Industrial' else 15000
        vab_real = vab_base - (demanda_maxima * 2)
        
        datos.append([
            fecha.strftime('%Y-%m-%d'), fecha.year, fecha.month, (fecha.month-1)//3 + 1, 
            ['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo'][fecha.weekday()],
            provincias[provincia_idx], 'Sierra' if provincias[provincia_idx] in ['Pichincha', 'Azuay', 'Tungurahua', 'Loja'] else 'Costa',
            sector, empresas[provincia_idx], 
            round(energia_facturada, 4), round(demanda_maxima, 4), round(porcentaje_perdidas, 2),
            round(temperatura, 2), round(precipitacion, 2), round(vab_real, 4)
        ])
        
    columnas = ['Fecha', 'Anio', 'Mes', 'Trimestre', 'DiaSemana', 'Provincia', 'Region', 'Sector', 'Empresa',
                'EnergiaFacturada_MWh', 'DemandaMaxima_MW', 'PorcentajePerdidas',
                'Temperatura_C', 'Precipitacion_mm', 'VAB_MilesUSD']
    return pd.DataFrame(datos, columns=columnas)

df = generar_dataset_maestro()
print(f"Dataset generado con éxito. Total filas: {df.shape[0]}")

### 2. Inserción en Tablas de Dimensión
Llenamos primero los catálogos (Sectores, Provincias, Tiempos) para poder obtener sus IDs.

In [ ]:
try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    
    print("Poblando tablas de Dimensión...")
    
    # Dim_Sector
    for s in df['Sector'].unique():
        cursor.execute("IF NOT EXISTS (SELECT 1 FROM Dim_Sector WHERE NombreSector=?) INSERT INTO Dim_Sector (NombreSector) VALUES (?)", (s, s))
    
    # Dim_Geografia
    for _, row in df[['Provincia', 'Region']].drop_duplicates().iterrows():
        cursor.execute("IF NOT EXISTS (SELECT 1 FROM Dim_Geografia WHERE Provincia=?) INSERT INTO Dim_Geografia (Provincia, Region) VALUES (?, ?)", (row['Provincia'], row['Provincia'], row['Region']))
        
    # Dim_EmpresaElectrica
    for e in df['Empresa'].unique():
        cursor.execute("IF NOT EXISTS (SELECT 1 FROM Dim_EmpresaElectrica WHERE NombreEmpresa=?) INSERT INTO Dim_EmpresaElectrica (NombreEmpresa) VALUES (?)", (e, e))

    # Dim_Tiempo
    for _, row in df[['Fecha', 'Anio', 'Mes', 'Trimestre', 'DiaSemana']].drop_duplicates().iterrows():
        cursor.execute("IF NOT EXISTS (SELECT 1 FROM Dim_Tiempo WHERE Fecha=?) INSERT INTO Dim_Tiempo (Fecha, Anio, Mes, Trimestre, DiaSemana) VALUES (?, ?, ?, ?, ?)", 
                       (row['Fecha'], row['Fecha'], row['Anio'], row['Mes'], row['Trimestre'], row['DiaSemana']))
    
    conn.commit()
    print("Dimensiones pobladas correctamente.")
    
except Exception as e:
    print(f"Error: {e}")

### 3. Mapeo de IDs e Inserción Masiva de Hechos
Cruzamos las tablas de dimensión para obtener las PKs y enviamos las FKs a la tabla de hechos final.

In [ ]:
try:
    print("Mapeando llaves primarias...")
    dict_sector = dict(pd.read_sql("SELECT IdSector, NombreSector FROM Dim_Sector", conn).values)
    dict_geo = dict(pd.read_sql("SELECT IdGeografia, Provincia FROM Dim_Geografia", conn).values)
    dict_emp = dict(pd.read_sql("SELECT IdEmpresa, NombreEmpresa FROM Dim_EmpresaElectrica", conn).values)
    dict_tiempo = dict(pd.read_sql("SELECT IdTiempo, CONVERT(VARCHAR, Fecha, 23) as Fecha FROM Dim_Tiempo", conn).values)
    
    inv_sector = {v: k for k, v in dict_sector.items()}
    inv_geo = {v: k for k, v in dict_geo.items()}
    inv_emp = {v: k for k, v in dict_emp.items()}
    inv_tiempo = {v: k for k, v in dict_tiempo.items()}

    print("Preparando registros de Hechos...")
    fact_demanda = []
    fact_clima = []
    fact_economia = []
    
    for _, row in df.iterrows():
        id_t = inv_tiempo.get(row['Fecha'])
        id_g = inv_geo.get(row['Provincia'])
        id_s = inv_sector.get(row['Sector'])
        id_e = inv_emp.get(row['Empresa'])
        
        if id_t and id_g and id_s and id_e:
            fact_demanda.append((id_t, id_g, id_s, id_e, row['EnergiaFacturada_MWh'], row['DemandaMaxima_MW'], row['PorcentajePerdidas']))
            fact_clima.append((id_t, id_g, row['Temperatura_C'], row['Precipitacion_mm']))
            fact_economia.append((id_t, id_s, row['VAB_MilesUSD']))

    print(f"Insertando {len(fact_demanda)} registros en la base de datos...")
    
    cursor.executemany("INSERT INTO Fact_Demanda (IdTiempo, IdGeografia, IdSector, IdEmpresa, EnergiaFacturada_MWh, DemandaMaxima_MW, PorcentajePerdidas) VALUES (?, ?, ?, ?, ?, ?, ?)", fact_demanda)
    cursor.executemany("INSERT INTO Fact_Clima (IdTiempo, IdGeografia, Temperatura_C, Precipitacion_mm) VALUES (?, ?, ?, ?)", fact_clima)
    cursor.executemany("INSERT INTO Fact_Economia (IdTiempo, IdSector, VAB_MilesUSD) VALUES (?, ?, ?)", fact_economia)

    conn.commit()
    print("\n\n--------------------------------------------------------------")
    print("EXITO: Inyeccion masiva completada.")
    print("Tu base de datos SQL Server tiene toda la data lista para ML.")
    print("--------------------------------------------------------------")
    
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals(): conn.close()
